In [ ]:
vocab = "$abcdefghijklmnopqrstuvwxyz"
vocab_size = len(vocab)
print(vocab_size)
ch_to_i = {char: i for i, char in enumerate(vocab)}
i_to_ch = {i: char for i, char in enumerate(vocab)}

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

equal_probs = F.softmax(torch.ones(vocab_size), dim=0)
for i in range(5):
    generated = ""
    while True:
        rand_int = torch.multinomial(equal_probs, 1).item()
        rand_char = i_to_ch[rand_int]
        if rand_char == "$":
            break

        generated += rand_char

    print(f"name {i}: {generated}")

In [ ]:
names = []
# edits each name into $<name>$  
with open('data/names_2022.txt', 'r') as file:
    for line in file:
        name, _, _= line.lower().strip().split(',')
        names.append("$" + name + "$")
len(names)


In [ ]:
bigram = torch.zeros((vocab_size, vocab_size))
total = 0
for name in names:
    for ch1, ch2 in zip(name, name[1:]):
        ch1_int = ch_to_i[ch1]
        ch2_int = ch_to_i[ch2]
        bigram[ch1_int][ch2_int] += 1
        total += 1
bigram /= total
    

In [ ]:
for i in range(5):
    generated = "$"
    while True:
        bigram_probs = bigram[ch_to_i[generated[-1]]]
        sampled_char = i_to_ch[
            torch.multinomial(bigram_probs, 1).item()
        ]
        if sampled_char == "$":
            break
        generated += sampled_char
    print(f"name {i}: {generated[1:]}")


In [ ]:
example_name = "$ada$"
encode = lambda word: torch.tensor([ch_to_i[c] for c in word])
decode = lambda tensor_i: ''.join(i_to_ch[i.item()] for i in tensor_i)
print(encode(example_name))
print(decode(encode(example_name)))

name_indices = [encode(name) for name in names]
target_indices = [name_index[1:] for name_index in name_indices]

In [ ]:
# Making name and target indices the same lentgh = max_name_length

from torch.nn.utils.rnn import pad_sequence
X = pad_sequence(name_indices, batch_first=True, padding_value=0)
max_name_length = max(len(name) for name in names)
target_indices.append(torch.empty((max_name_length), dtype=torch.long)) # adds a new dummy tensor into target_indices so that max lenght of target will be 11
Y = pad_sequence(target_indices, batch_first=True, padding_value=-1)[:-1]
print(X[0])
print(Y[0])

In [ ]:
def get_batch(batch_size=64):
    """creates a random batch of input and labels"""
    random_idx = torch.randint(0, X.size(0), (batch_size,)) # creating a random index with dimensions of the padded sequence and the total amount of names. eg; 31915
    # print(X.shape) 
    # print(X.size(0)) 
    # print('random_idx ', random_idx)
    inputs = X[random_idx]
    labels = Y[random_idx]
    return inputs, labels
inputs, labels = get_batch(batch_size=3)
print(inputs)
print(inputs.shape)
print(labels)


In [ ]:
embedding_dim = 3 # arbitrary
embedding = nn.Embedding(vocab_size, embedding_dim)
example_input = torch.tensor([1,1,0,2])
input_emb = embedding(example_input)
print(input_emb.shape) # 4 input values so 4 rows for 4 embeddings with 3 dimensions (defined)
input_emb

In [ ]:
class SequenceMLP(nn.Module):
    def __init__(self, vocab_size, max_sequence_length, embedding_dim, hidden_dim=32):
        super().__init__()
        self.vocab_size = vocab_size # 27 
        self.max_sequence_length = max_sequence_length # 17
        self.embedding_dim = embedding_dim # 3
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.linear = nn.Linear(embedding_dim*max_sequence_length, hidden_dim) # flattening the padded matrix into a vector
        self.relu = nn.ReLU()
        self.out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        batch_size, seq_len = x.shape # x = inputs; here: batch_size=64; seq_len is 17 
        sequence_embeddings = torch.zeros(
            batch_size, 
            seq_len, 
            self.max_sequence_length*self.embedding_dim # 17*3=51
        ) 

        for i in range(seq_len): # seq_len is 17 here
            subsequence = torch.zeros(
                batch_size, # 64
                self.max_sequence_length, # 17
                dtype=torch.int
            )
            prefix = x[:, :i+1] # slices and selects up to the i+1th index
            subsequence[:, :i+1] = prefix # stores the same thing as prefix with the rest of the values being zero
            emb = self.embedding(subsequence)  # look up the embedding value for each element in the subsequence and store it; (64,17,3)
            sequence_embeddings[:, i, :] = emb.view(batch_size, -1) # creating shape 64,17x3=64,51 for each ith row in each matrix 
        x = self.linear(sequence_embeddings)
        x = self.relu(x)
        x = self.out(x)
        return x

embedding_dim = 3
max_sequence_length = X.shape[1] # 17
model = SequenceMLP(vocab_size, max_sequence_length, embedding_dim)



In [ ]:
# Demo: for ONE example, show the triangular `subsequence` built inside
# SequenceMLP.forward, and the flattened embedding it produces at each step.
# (No training here — just replaying the loop body once, on one padded name.)

example = X[0].unsqueeze(0)  # one padded name, batch_size=1 -> shape (1, 17)
print(f"example input (padded): {example}")
print(f"decoded: {decode(example[0])}")

batch_size, seq_len = example.shape
steps_to_show = sorted(set([0, 1, 2, 5, seq_len - 1]))  # only print a few rows of the triangle

for i in range(seq_len):
    subsequence = torch.zeros(batch_size, model.max_sequence_length, dtype=torch.int)
    prefix = example[:, :i + 1]
    subsequence[:, :i + 1] = prefix
    emb = model.embedding(subsequence)
    flattened = emb.view(batch_size, -1)

    if i in steps_to_show:
        print(f"\n--- i={i} ---")
        print(f"prefix = x[:, :{i+1}]                 -> shape={tuple(prefix.shape)}  values={prefix.tolist()}")
        print(f"subsequence (triangle row {i})         -> shape={tuple(subsequence.shape)}")
        print(subsequence)
        print(f"emb = embedding(subsequence)           -> shape={tuple(emb.shape)}")
        print(f"flattened = emb.view(batch_size, -1)   -> shape={tuple(flattened.shape)}")
        print(flattened)


In [ ]:
import torch.optim as optim

def train(model, optimizer, num_steps=10_001, loss_report_interval=1_000):
    losses = []
    for i in range(1, num_steps):
        inputs, labels = get_batch() # gets the inputs and labels randomly created with get_batch()
        optimizer.zero_grad()
        logits = model(inputs)
        loss = F.cross_entropy(
            input=logits.view(-1, logits.shape[-1]),
            target=labels.view(-1), 
            ignore_index=-1
        )    
        losses.append(loss.item())
        if i % loss_report_interval == 0:
            print(f'Average loss at step {i}: {sum(losses[
                -loss_report_interval:]) / loss_report_interval:.4f}')
        loss.backward()
        optimizer.step()

optimizer = optim.SGD(model.parameters(), lr=0.1)

In [ ]:
train(model, optimizer)

In [ ]:
x = torch.tensor((1,2,3,4,5))
for i in range(4):
    print(x[:i+1])
x[:]

In [ ]:
y = torch.randn(2,3,4)
y

In [ ]:
z =torch.zeros(3,2)
z


In [ ]:
x = torch.randn(3,4,3)
print(x)
x[:,1,:] = torch.randn(3)
x


In [ ]:
y = torch.randn(3,2)
model_test = nn.Linear(2,3)
print("y: ", y)
print("Linear output", model_test(y))


In [ ]:
torch.manual_seed(0)
x = torch.rand(2,3)

query = F.linear(x, weight=torch.rand(3,3), bias=torch.rand(3))
key = F.linear(x, weight=torch.rand(3,3), bias=torch.rand(3))
value = F.linear(x, weight=torch.rand(3,3), bias=torch.rand(3))

def dot_product_attention_single(q, k ,v):
    attn_weights = q @ k.T
    attn_weights = F.softmax(attn_weights, dim=-1)
    output = attn_weights @ v
    return output

dot_product_attention_single(query, key, value)

In [ ]:
torch.manual_seed(0)
x = torch.rand(1, 2, 3)

query = F.linear(x, weight=torch.rand(3, 3), bias=torch.rand(3))
key = F.linear(x, weight=torch.rand(3, 3), bias=torch.rand(3))
value = F.linear(x, weight=torch.rand(3, 3), bias=torch.rand(3))

def scaled_dot_product_causal_attention(q, k, v):
    attn_weights = q @ k.transpose(1, 2)
    mask = torch.tril(torch.ones(attn_weights.
    shape[1:]), diagonal=0)
    attn_weights = attn_weights.masked_fill(mask == 0, value=float('-inf'))
    attn_weights = attn_weights / torch.sqrt(torch.tensor(
    k.shape[-1]).float())
    attn_weights = F.softmax(attn_weights, dim=-1)
    output = attn_weights @ v
    return output, attn_weights

output, attn_weights = scaled_dot_product_causal_attention(query, key, value)
output


In [ ]:
from numpy import diagonal


class AttentionMLP(nn.Module):
    def __init__(self, n_embd, vocab_size, block_size, n_hidden=64):
        super().__init__()
        self.tok_embd = nn.Embedding(vocab_size, n_embd)
        self.attn_weights = None

        self.query_proj = nn.Linear(n_embd, n_embd)
        self.key_proj = nn.Linear(n_embd, n_embd)
        self.value_proj = nn.Linear(n_embd, n_embd)

        self.register_buffer("mask", torch.tril(torch.ones((block_size, block_size)), diagonal=0))

        self.mlp = nn.Sequential(
            nn.Linear(n_embd, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, n_embd)
        )

        self.output_proj = nn.Linear(n_embd, vocab_size)

    def forward(self, x):
        x = self.tok_embd(x)
        batch_size, seq_len, embd_dim = x.shape

        q = self.query_proj(x)
        k = self.key_proj(x)
        v = self.value_proj(x)

        attn_weights = q @ k.transpose(1, 2)
        attn_weights = attn_weights.masked_fill(self.mask[
        :seq_len, :seq_len] == 0, value=float('-inf'))
        attn_weights = attn_weights / torch.sqrt(torch.tensor(
        k.shape[-1]).float())
        self.attn_weights = F.softmax(attn_weights, dim=-1)
        x = self.attn_weights @ v
        x = self.mlp(x)

        x = self.output_proj(x)
        return x

model = AttentionMLP(
    32, 
    vocab_size, 
    max_name_length
)
optimizer = optim.SGD(model.parameters(), lr=0.01)
train(model, optimizer, num_steps=10_001, loss_report_interval=1_000)


In [ ]:
def generate_samples(model, num_samples=1, max_len=max_name_length):
    sequences = torch.zeros((num_samples, 1)).int()
    for _ in range(max_len):
        logits = model(sequences)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        sequences = torch.cat((sequences, idx_next), dim=1)
    for sequence in sequences:
        indices = torch.where(sequence == 0)[0]
        end = indices[1] if len(indices) > 1 else max_len
        sequence = sequence[1:end]
        print(decode(sequence))

generate_samples(model, num_samples=10)




In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, n_embd, num_heads=4, n_hidden=64):
        super().__init__()
        assert n_embd % num_heads == 0, "Embedding dimension must be divisible by the number of heads"

        self.num_heads = num_heads
        self.head_dim = n_embd // num_heads

        self.query_proj = nn.Linear(n_embd, n_embd)
        self.key_proj = nn.Linear(n_embd, n_embd)
        self.value_proj = nn.Linear(n_embd, n_embd)

        self.mlp = nn.Sequential(
            nn.Linear(n_embd, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, n_embd)
        )

        # Layernorms
        self.norm_1 = nn.LayerNorm(n_embd)
        self.norm_2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        batch_size, sequence_length, _ = x.shape

        q = self.query_proj(x)
        k = self.key_proj(x)
        v = self.value_proj(x)

        # multiheaded attention
        q = q.view(batch_size, sequence_length, self.num_heads,
         self.head_dim).transpose(1, 2)
        k = k.view(batch_size, sequence_length, self.num_heads,
         self.head_dim).transpose(1, 2)
        v = v.view(batch_size, sequence_length, self.num_heads,
         self.head_dim).transpose(1, 2)

        # attention
        attn_weights = F.scaled_dot_product_attention(
        q, k, v, is_causal=True)

        # multiple heads concatenation
        attn_weights = attn_weights.transpose(1, 2).contiguous().view(
        batch_size, sequence_length, -1)

        # norm and residual connections here
        x = self.norm_1(x + attn_weights)
        x = self.norm_2(x + self.mlp(x))
        return x

In [ ]:
class Transformer(nn.Module):
    def __init__(self, n_embd, vocab_size, block_size, num_blocks=6):
        super().__init__()
        self.char_embedding = nn.Embedding(vocab_size, n_embd)
        self.positional_embedding = nn.Embedding(block_size, n_embd)

        self.transformer_blocks = nn.Sequential(
            *[TransformerBlock(n_embd) for _ in range(num_blocks)]
        )

        self.output_proj = nn.Linear(n_embd, vocab_size)

    def forward(self, x):
        _, seq_len = x.shape

        pos_embd = self.positional_embedding(torch.arange(seq_len))
        char_embd = self.char_embedding(x)
        x = char_embd + pos_embd
        x = self.transformer_blocks(x)
        x = self.output_proj(x)

        return x

n_embd = 64
model = Transformer(n_embd, vocab_size, block_size=max_name_length)
optimizer = optim.SGD(model.parameters(), lr=0.1)
train(model, optimizer, num_steps=10_001, loss_report_interval=1_000)
